# Rapport SHAP — Mots influents sur le score de détresse

**Artefact School of Data — Bootcamp Data Science, Mars 2026**

Ce notebook génère le graphique des mots les plus influents du modèle baseline (TF-IDF + LR) :
- **Rouge** → pousse vers la détresse
- **Bleu** → pousse vers non-détresse

Valeurs SHAP linéaires : `tfidf(mot) × coef_LR(mot)` — interprétables sans boîte noire.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib

# Racine du projet
def _find_root():
    p = Path().resolve()
    for _ in range(6):
        if (p / 'src').exists() and (p / 'models').exists():
            return p
        p = p.parent
    raise FileNotFoundError('Racine du projet introuvable')

ROOT = _find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT))

REPORTS_DIR = ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)
print('ROOT :', ROOT)
print('Reports :', REPORTS_DIR)


In [ ]:
# Chargement du modèle baseline
model_path = ROOT / 'models' / 'baseline.joblib'
if not model_path.exists():
    model_path = ROOT / 'models' / 'baseline.pkl'

pipeline = joblib.load(model_path)
vectorizer = pipeline.named_steps['tfidf']
clf        = pipeline.named_steps['clf']

feature_names = vectorizer.get_feature_names_out()
coefs         = clf.coef_[0]   # coefficients LR pour classe 1 (détresse)

print(f'Modèle chargé : {model_path.name}')
print(f'Vocabulaire   : {len(feature_names):,} mots')
print(f'Classes       : {clf.classes_}  (0=neutre, 1=détresse)')


In [ ]:
# ── Top N mots globaux (coefficients LR bruts) ────────────────────────────────
N = 20   # nombre de mots par sens

top_pos_idx = np.argsort(coefs)[::-1][:N]   # plus fort vers détresse
top_neg_idx = np.argsort(coefs)[:N]          # plus fort vers non-détresse

words  = np.concatenate([feature_names[top_neg_idx], feature_names[top_pos_idx]])
values = np.concatenate([coefs[top_neg_idx], coefs[top_pos_idx]])

df_global = pd.DataFrame({'word': words, 'coef': values}).sort_values('coef')
df_global.head()


In [ ]:
# ── Graphique global — Top mots influents (coefficients LR) ──────────────────
fig, ax = plt.subplots(figsize=(9, max(6, len(df_global) * 0.38)))

colors = ['#d73027' if v > 0 else '#4575b4' for v in df_global['coef']]
ax.barh(df_global['word'], df_global['coef'], color=colors, edgecolor='white', linewidth=0.4)
ax.axvline(0, color='black', linewidth=0.9)

for bar, val in zip(ax.patches, df_global['coef']):
    x = bar.get_width()
    ax.text(x + (0.01 if x >= 0 else -0.01), bar.get_y() + bar.get_height() / 2,
            f'{val:+.2f}', va='center', ha='left' if x >= 0 else 'right', fontsize=8)

ax.set_xlabel('Coefficient LR (contribution au score de détresse)', fontsize=10)
ax.set_title(f'Top {N} mots influents — Modèle TF-IDF + LR\n(Rouge = détresse · Bleu = non-détresse)', fontsize=11, fontweight='bold')

red_patch  = mpatches.Patch(color='#d73027', label='Pousse vers détresse')
blue_patch = mpatches.Patch(color='#4575b4', label='Pousse vers non-détresse')
ax.legend(handles=[red_patch, blue_patch], loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)

fig.tight_layout()
out = REPORTS_DIR / 'shap_top_words_global.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print('Sauvegardé :', out)
plt.show()


In [ ]:
# ── SHAP sur texte personnalisé ───────────────────────────────────────────────
from src.training.preprocess import clean_text

SAMPLE_TEXTS = [
    "I feel so hopeless and empty, nothing makes sense anymore, I want to die",
    "I had a great day, went hiking with friends and felt really alive",
    "I can't sleep, I'm exhausted but my mind won't stop, I feel like a burden",
]

def shap_for_text(text: str, n_features: int = 15):
    cleaned    = clean_text(text)
    X          = vectorizer.transform([cleaned])
    tfidf_vals = X.toarray()[0]
    contribs   = tfidf_vals * coefs
    nonzero    = np.where(tfidf_vals > 0)[0]
    if nonzero.size == 0:
        return pd.DataFrame(columns=['word', 'tfidf', 'shap_value'])
    top_idx = nonzero[np.argsort(np.abs(contribs[nonzero]))[::-1]][:n_features]
    return pd.DataFrame({
        'word':       feature_names[top_idx],
        'tfidf':      tfidf_vals[top_idx].round(3),
        'shap_value': contribs[top_idx].round(3),
    }).sort_values('shap_value')

fig, axes = plt.subplots(len(SAMPLE_TEXTS), 1, figsize=(10, 5 * len(SAMPLE_TEXTS)))

for ax, text in zip(axes, SAMPLE_TEXTS):
    df  = shap_for_text(text)
    proba = pipeline.predict_proba([clean_text(text)])[0]
    score = proba[1]

    if df.empty:
        ax.text(0.5, 0.5, 'Aucun mot reconnu', ha='center', va='center')
        continue

    colors = ['#d73027' if v > 0 else '#4575b4' for v in df['shap_value']]
    ax.barh(df['word'], df['shap_value'], color=colors, edgecolor='white', linewidth=0.3)
    ax.axvline(0, color='black', linewidth=0.8)

    for bar, row in zip(ax.patches, df.itertuples()):
        x = bar.get_width()
        label = f'{row.tfidf:.2f} = {row.word}'
        ax.text(x + (0.005 if x >= 0 else -0.005), bar.get_y() + bar.get_height() / 2,
                f'{row.shap_value:+.2f}', va='center', ha='left' if x >= 0 else 'right', fontsize=8)

    ax.set_title(f'"{text[:70]}..."\nScore détresse : {score:.2f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Contribution SHAP')
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('SHAP par texte — Modèle TF-IDF + LR', fontsize=12, fontweight='bold', y=1.01)
fig.tight_layout()
out2 = REPORTS_DIR / 'shap_per_text.png'
fig.savefig(out2, dpi=150, bbox_inches='tight')
print('Sauvegardé :', out2)
plt.show()


## Rapports générés
- `reports/shap_top_words_global.png` — Top 20 mots globaux (coefficients LR)
- `reports/shap_per_text.png` — SHAP sur 3 exemples de textes

Pour personnaliser les textes analysés, modifie la liste `SAMPLE_TEXTS` dans la cellule précédente.